# Поиск по статьям поддержки

Ноутбук помогает загрузить утверждённый корпус JSONL, изучить фрагменты и вручную посмотреть результаты lexical retrieval. Сейчас в репозитории нет статей, поэтому сначала добавьте их по формату из `data/knowledge/README.md`. Значение сходства не является вероятностью правильного ответа.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CORPUS_PATH = ROOT / 'data' / 'knowledge' / 'articles.jsonl'
print(f'Корень проекта: {ROOT}')
print(f'Корпус: {CORPUS_PATH}')
print(f'Файл существует: {CORPUS_PATH.is_file()}')


In [ ]:
if not CORPUS_PATH.is_file():
    raise FileNotFoundError(
        'Добавьте одобренные статьи в data/knowledge/articles.jsonl. '         'Формат описан в data/knowledge/README.md.'
    )

from src.rag.retriever import KnowledgeRetriever
retriever = KnowledgeRetriever.from_jsonl(CORPUS_PATH)
print(f'Проиндексировано фрагментов: {len(retriever.chunks)}')
print(f'Число документов: {len({chunk.document_id for chunk in retriever.chunks})}')


In [ ]:
query = 'card payment reversed'
hits = retriever.search(query, top_k=5)
for rank, hit in enumerate(hits, start=1):
    print(f'{rank}. {hit.title}')
    print(f'   Документ: {hit.document_id} | фрагмент: {hit.chunk_id}')
    print(f'   Язык: {hit.language} | актуальность: {hit.updated_at}')
    print(f'   Источник: {hit.source}')
    print(f'   Лексическое сходство: {hit.lexical_similarity:.4f}')
    print(f'   Фрагмент: {hit.text}\n')


## Как читать результат

Список упорядочен по лексическому сходству запроса и фрагмента. Низкая или высокая оценка сама по себе не задаёт порог релевантности. Перед автоматической генерацией нужно подготовить экспертно размеченные запросы с релевантными ID документов и посчитать Recall@5 и MRR. Включите запросы без подходящей статьи, чтобы спроектировать отказ и передачу оператору.


## Оценка на размеченных запросах

Подготовьте `data/knowledge/queries.jsonl` по формату из README базы знаний. Укажите все релевантные ID для каждого запроса. Пустой список означает, что ответа в текущем корпусе нет. Не включайте демонстрационные запросы в отчёт о качестве.


In [ ]:
QUERY_PATH = ROOT / 'data' / 'knowledge' / 'queries.jsonl'
if not QUERY_PATH.is_file():
    print('Размеченная выборка отсутствует. Добавьте queries.jsonl, чтобы посчитать метрики.')
else:
    from src.rag.evaluation import evaluate_retrieval, load_queries
    examples = load_queries(QUERY_PATH)
    evaluation = evaluate_retrieval(retriever, examples, k=5)
    print(f"Запросов всего: {evaluation['query_count']}")
    print(f"С релевантными статьями: {evaluation['answerable_query_count']}")
    print(f"Без ответа в корпусе: {evaluation['unanswerable_query_count']}")
    print(f"Recall@5: {evaluation['recall_at_5']:.4f}")
    print(f"MRR: {evaluation['mrr']:.4f}")


In [ ]:
if QUERY_PATH.is_file():
    import pandas as pd
    pd.DataFrame(evaluation['answerable_query_results'])


Метрики считаются только по запросам, размеченным как имеющие релевантные статьи. Recall@5 — средняя доля релевантных документов в первых пяти результатах, MRR — средний обратный ранг первого релевантного документа. Запросы без статьи не входят в эти метрики: порог отказа и корректную передачу оператору нужно оценивать отдельно.
